# AMPS II · Week 05 — HW5
## Viscoelastic time integration and correspondence, phase-field mesh and length-scale studies, and an LLM-generated phase-field script

**Due before Week 6 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be wrong.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Time integration of a Maxwell branch, the creep compliance of the standard linear solid, AT2 strength and length scale (theory) | 20 |
| 2 | Viscoelasticity: the order of the time schemes in 2D, relaxation under displacement control, branches with different Poisson ratios | 25 |
| 3 | Phase field: mesh and length-scale study of the slit specimen, the split in compression | 35 |
| 4 | Debugging an LLM-generated phase-field script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run the three cells once.

In [ ]:
%%writefile /tmp/fnx/visco.py
"""Linear viscoelasticity, generalized Maxwell model (Prony series) -- NumPy only.
E(t) = E_inf + sum_i E_i exp(-t/tau_i);  branch i: d(eps_v_i)/dt = (eps - eps_v_i)/tau_i,  sigma = E_inf eps + sum_i E_i (eps - eps_v_i).
One time step of a branch, in the common form   eps_v_new = a eps_v + (1 - b) eps_new + (b - a) eps_old:
  backward Euler (BE):           a = b = tau/(tau + dt)
  exponential (EXP, exact for a strain that is linear in time over the step):  a = exp(-dt/tau),  b = (1 - a) tau/dt
  an instantaneous step (dt = 0) is elastic: a = b = 1."""
import numpy as np

class Prony:
    def __init__(self, E_inf, branches, nu=0.3):
        """branches = [(E_1, tau_1), (E_2, tau_2), ...]; nu: the same Poisson ratio for every branch"""
        self.E_inf, self.br, self.nu = float(E_inf), [(float(E), float(t)) for E, t in branches], nu
        self.E0 = self.E_inf + sum(E for E, _ in self.br)       # instantaneous (glassy) modulus

    def relax(self, t):
        """relaxation modulus E(t), exact"""
        t = np.asarray(t, dtype=float)
        return self.E_inf + sum(E * np.exp(-t / tau) for E, tau in self.br)

    def creep(self, t):
        """creep compliance J(t), exact: Laplace transform  J(s) = 1 / (s^2 E(s)),  inverted by residues"""
        P = np.polynomial.polynomial
        D = np.array([1.0])
        for _, tau in self.br:
            D = P.polymul(D, [1 / tau, 1.0])                   # prod (s + 1/tau_i)
        N = self.E_inf * D                                     # s E(s) = N(s) / D(s)
        for k, (E, tau) in enumerate(self.br):
            others = np.array([1.0])
            for j, (_, tj) in enumerate(self.br):
                if j != k:
                    others = P.polymul(others, [1 / tj, 1.0])
            N = P.polyadd(N, E * P.polymul([0.0, 1.0], others))
        roots = P.polyroots(N)                                 # -1/lambda_k: retardation
        dN = P.polyder(N)
        t = np.asarray(t, dtype=float)
        J = P.polyval(0.0, D) / P.polyval(0.0, N) * np.ones_like(t)
        for r in roots:
            J = J + np.real(P.polyval(r, D) / (r * P.polyval(r, dN)) * np.exp(r * t))
        return J

    def coeffs(self, dt, scheme="exp"):
        """(a_i, b_i) of every branch for a step dt"""
        out = []
        for _, tau in self.br:
            if dt == 0:
                out.append((1.0, 1.0))
            elif scheme == "be":
                th = tau / (tau + dt); out.append((th, th))
            else:
                a = np.exp(-dt / tau); out.append((a, (1 - a) * tau / dt))
        return out

def point(mat, t, eps=None, sig=None, scheme="exp", jump=True):
    """one material point in 1D. Give a strain history eps(t) (relaxation) or a stress history sig(t) (creep), as callables.
    t: times t_0 = 0 < t_1 < ...; jump=True: the load at t = 0 is applied as an instantaneous (elastic) step at t = 0+.
    Returns (eps_n, sig_n) at t_1, t_2, ... (and at 0+ as the first entry if jump)."""
    ev = np.zeros(len(mat.br)); e_old = 0.0; out = []
    steps = ([0.0] if jump else []) + list(np.diff(t))
    times = ([t[0]] if jump else []) + list(t[1:])
    for dt, tn in zip(steps, times):
        ab = mat.coeffs(dt, scheme)
        if eps is not None:
            e = eps(tn)
        else:                                                   # stress control: solve the (linear) step for eps
            num = sig(tn) + sum(E * (a * v + (b - a) * e_old) for (E, _), (a, b), v in zip(mat.br, ab, ev))
            e = num / (mat.E_inf + sum(E * b for (E, _), (a, b) in zip(mat.br, ab)))
        ev = np.array([a * v + (1 - b) * e + (b - a) * e_old for (a, b), v in zip(ab, ev)])
        s = mat.E_inf * e + sum(E * (e - v) for (E, _), v in zip(mat.br, ev))
        out.append((e, s)); e_old = e
    return np.array(out)

In [ ]:
%%writefile /tmp/fnx/vtube.py
"""A viscoelastic thick tube (plane strain) under inner pressure or inner radial displacement, in DOLFINx 0.11.
Generalized Maxwell in 3D: C(t) = C_inf + sum_i C_i exp(-t/tau_i), each C_i isotropic with its own (E_i, nu_i).
The viscous strains of the branches live at quadrature points (Mandel vectors, as in Week 4); every step is a LINEAR problem
with the effective stiffness C_inf + sum b_i C_i (factorised once per step size) and a history load on the right-hand side."""
import sys
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc
sys.path.insert(0, "/tmp/fnx")
from visco import Prony

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, LEFT = 1, 2, 3, 4
ONE = np.array([1.0, 1.0, 1.0, 0.0])

def C4(E, nu):
    """isotropic stiffness, Mandel 4 x 4"""
    mu, lm = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))
    return lm * np.outer(ONE, ONE) + 2 * mu * np.eye(4)

def lame_ur(p, ri, ro, E, nu):
    """elastic plane-strain u_r(ri) under inner pressure p"""
    A = p * ri**2 / (ro**2 - ri**2); B = A * ro**2
    return (1 + nu) / E * ((1 - 2 * nu) * A * ri + B / ri)

def annulus_mesh(ri, ro, nr, nt):
    """curved quarter annulus (6-node triangles mapped from (r, theta)), as Weeks 3-4. Tags 1 r = ri, 2 r = ro, 3 y = 0, 4 x = 0."""
    if comm.rank == 0:
        nI, nJ = 2 * nr + 1, 2 * nt + 1
        R, TH = np.meshgrid(np.linspace(ri, ro, nI), np.linspace(0, np.pi / 2, nJ), indexing="ij")
        x = np.c_[(R * np.cos(TH)).ravel(), (R * np.sin(TH)).ravel()]
        x[np.abs(x) < 1e-14] = 0.0
        node = lambda p: p[0] * nJ + p[1]
        mid = lambda p, q: ((p[0] + q[0]) // 2, (p[1] + q[1]) // 2)
        cells = []
        for a in range(nr):
            for b in range(nt):
                i, j = 2 * a, 2 * b
                for p, q, s in (((i, j), (i + 2, j), (i + 2, j + 2)), ((i, j), (i + 2, j + 2), (i, j + 2))):
                    cells.append([node(p), node(q), node(s), node(mid(q, s)), node(mid(p, s)), node(mid(p, q))])
        cells = np.array(cells, dtype=np.int64)
    else:
        x, cells = np.zeros((0, 2)), np.zeros((0, 6), dtype=np.int64)
    msh = mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))), x)
    rad = lambda X: np.hypot(X[0], X[1])
    fac, val = [], []
    for tag, f in ((INNER, lambda X: np.isclose(rad(X), ri)), (OUTER, lambda X: np.isclose(rad(X), ro)),
                   (BOTTOM, lambda X: np.isclose(X[1], 0.0)), (LEFT, lambda X: np.isclose(X[0], 0.0))):
        e = mesh.locate_entities_boundary(msh, 1, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return msh, mesh.meshtags(msh, 1, fac, np.concatenate(val)[idx])

class ViscoTube:
    """mat: Prony (common nu) -- or give branch_nu = [nu_inf, nu_1, nu_2, ...] for different Poisson ratios per branch.
    control = "pressure" or "displacement" (inner radial displacement). scheme = "exp" or "be"."""
    def __init__(self, msh, ft, mat, ri, ro, control="pressure", scheme="exp", branch_nu=None, qdeg=2):
        self.msh, self.mat, self.ri, self.ro, self.control, self.scheme = msh, mat, ri, ro, control, scheme
        nus = branch_nu or [mat.nu] * (1 + len(mat.br))
        self.Cinf = C4(mat.E_inf, nus[0]); self.Cb = [C4(E, n) for (E, _), n in zip(mat.br, nus[1:])]
        V = self.V = fem.functionspace(msh, ("Lagrange", 2, (2,)))
        cell = msh.basix_cell()
        Q4 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4,), degree=qdeg))
        Q44 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4, 4), degree=qdeg))
        self.ev = [fem.Function(Q4) for _ in mat.br]            # viscous strains
        self.e_old, self.e_new, self.sig = fem.Function(Q4), fem.Function(Q4), fem.Function(Q4)
        self.hist = fem.Function(Q4)                            # history stress  sum C_i (a eps_v + (b - a) eps_old)
        self.Ceff = fem.Function(Q44)
        self.u = fem.Function(V)
        dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": qdeg})
        ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
        n = ufl.FacetNormal(msh)
        du, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        self.p = fem.Constant(msh, 0.0)
        self.a = fem.form(ufl.dot(ufl.dot(self.Ceff, self.eps4(du)), self.eps4(v)) * dx)
        L = ufl.dot(self.hist, self.eps4(v)) * dx
        if control == "pressure":
            L = L - self.p * ufl.dot(n, v) * ds(INNER)
        self.L = fem.form(L)
        self.e_expr = fem.Expression(self.eps4(self.u), Q4.element.interpolation_points)
        self.bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 1, ft.find(tag)), V.sub(c)) for tag, c in ((BOTTOM, 1), (LEFT, 0))]
        self.g = fem.Function(V)
        if control == "displacement":
            self.bcs.append(fem.dirichletbc(self.g, fem.locate_dofs_topological(V, 1, ft.find(INNER))))
        wR = fem.Function(V)
        wR.interpolate(lambda X: np.vstack([X[0], X[1]]) / np.hypot(X[0], X[1]) * (ro - np.hypot(X[0], X[1])) / (ro - ri))
        self.R_form = fem.form(ufl.dot(self.sig, self.eps4(wR)) * dx)
        self.ur_form = fem.form(ufl.dot(self.u, -n) * ds(INNER))
        self.b = create_vector(V); self.A = None; self.dt_A = None
        self.ksp = PETSc.KSP().create(comm); self.ksp.setType("preonly")
        self.ksp.getPC().setType("lu"); self.ksp.getPC().setFactorSolverType("mumps")
        self.t = 0.0

    @staticmethod
    def eps4(u):
        e = ufl.sym(ufl.grad(u))
        return ufl.as_vector([e[0, 0], e[1, 1], 0, np.sqrt(2) * e[0, 1]])

    def step(self, dt, load):
        """advance by dt (dt = 0: instantaneous elastic step) to the load value (pressure or inner radial displacement)"""
        ab = self.mat.coeffs(dt, self.scheme)
        n4 = len(self.e_old.x.array) // 4
        if self.dt_A != dt:                                     # effective stiffness: reassemble only if dt changes
            C = self.Cinf + sum(b * Ci for (a, b), Ci in zip(ab, self.Cb))
            self.Ceff.x.array[:] = np.tile(C.ravel(), n4)
            self.A = assemble_matrix(self.a, bcs=self.bcs); self.A.assemble()
            self.ksp.setOperators(self.A); self.dt_A = dt
        eo = self.e_old.x.array.reshape(n4, 4)
        h = sum((a * ev.x.array.reshape(n4, 4) + (b - a) * eo) @ Ci for (a, b), ev, Ci in zip(ab, self.ev, self.Cb))
        self.hist.x.array[:] = np.asarray(h).ravel() if len(self.ev) else 0.0
        if self.control == "pressure":
            self.p.value = load
        else:
            self.g.interpolate(lambda X: load * np.vstack([X[0], X[1]]) / np.hypot(X[0], X[1]))
        with self.b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(self.b, self.L); apply_lifting(self.b, [self.a], [self.bcs])
        self.b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(self.b, self.bcs)
        self.ksp.solve(self.b, self.u.x.petsc_vec); self.u.x.scatter_forward()
        self.e_new.interpolate(self.e_expr); en = self.e_new.x.array.reshape(n4, 4)
        s = en @ self.Cinf
        for (a, b), ev, Ci in zip(ab, self.ev, self.Cb):
            evn = a * ev.x.array.reshape(n4, 4) + (1 - b) * en + (b - a) * eo
            ev.x.array[:] = evn.ravel(); s = s + (en - evn) @ Ci
        self.sig.x.array[:] = s.ravel(); self.e_old.x.array[:] = self.e_new.x.array
        self.t += dt
        return self.ksp.getConvergedReason()

    def ur_inner(self):
        return comm.allreduce(fem.assemble_scalar(self.ur_form), op=MPI.SUM) / (np.pi / 2 * self.ri)

    def pressure_from_reaction(self):
        return comm.allreduce(fem.assemble_scalar(self.R_form), op=MPI.SUM) / (np.pi / 2 * self.ri)

In [ ]:
%%writefile /tmp/fnx/pfield.py
"""Phase-field fracture (AT2) in DOLFINx 0.11: the regularised crack energy  Gc/2 int (d^2/ell + ell |grad d|^2),
degradation g(d) = (1 - d)^2 + k, irreversibility by the history field H = max over time of psi+ (Miehe), the "hybrid" form
(isotropic degradation in the u-problem, psi+ from a split drives d), alternate minimisation (staggered) with a convergence check.
Meshes: the 1D crack profile on an interval; the unit square with a traction-free slit (no Gmsh, the same nodes on every machine)."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_matrix, create_vector, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def profile_1d(Gc, ell, L, n):
    """d on [0, L] with d(0) = 1 (the crack at x = 0), d' (L) = 0, no load: the optimal AT2 profile. Returns the regularised
    energy of this half crack and its exact value Gc/2 tanh(L/ell) (-> Gc/2 for L >> ell), and the computed d"""
    msh = mesh.create_interval(comm, n, [0.0, L])
    V = fem.functionspace(msh, ("Lagrange", 1))
    d, s = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = fem.form(Gc / ell * d * s * ufl.dx + Gc * ell * d.dx(0) * s.dx(0) * ufl.dx)
    L_ = fem.form(fem.Constant(msh, 0.0) * s * ufl.dx)
    bc = fem.dirichletbc(1.0, fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0.0)), V)
    dh = _lu_solve(a, L_, [bc], fem.Function(V))
    G = comm.allreduce(fem.assemble_scalar(fem.form(Gc / 2 * (dh**2 / ell + ell * dh.dx(0)**2) * ufl.dx)), op=MPI.SUM)
    return G, Gc / 2 * np.tanh(L / ell), dh

def _lu_solve(a, L, bcs, f):
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    b = assemble_vector(L); apply_lifting(b, [a], [bcs]); b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
    k = PETSc.KSP().create(comm); k.setOperators(A); k.setType("preonly"); k.getPC().setType("lu"); k.getPC().setFactorSolverType("mumps")
    k.solve(b, f.x.petsc_vec); f.x.scatter_forward()
    return f

def slit_mesh(h, a=0.5, alpha=0.25):
    """unit square, P1 triangles, cell size h along the crack line y = 0.5 (x uniform; y graded by a cubic map, coarse near
    y = 0 and 1), and a traction-free slit y = 0.5, 0 <= x < a made by duplicating the nodes on it (no Gmsh)."""
    nx = int(round(1 / h)); ny = 2 * max(1, int(round(alpha / h / 2)))
    if comm.rank == 0:
        xs = np.linspace(0, 1, nx + 1); s_ = np.linspace(-1, 1, ny + 1)
        ys = 0.5 + 0.5 * (alpha * s_ + (1 - alpha) * s_**3)
        X, Y = np.meshgrid(xs, ys, indexing="ij")
        node = lambda i, j: i * (ny + 1) + j
        jm, ia = ny // 2, int(round(a * nx))
        lower = {i: (nx + 1) * (ny + 1) + i for i in range(ia)}           # copies: the lower face of the slit
        x = np.vstack([np.c_[X.ravel(), Y.ravel()], np.c_[xs[:ia], np.full(ia, 0.5)]])
        cells = []
        for i in range(nx):
            for j in range(ny):
                q = [node(i, j), node(i + 1, j), node(i + 1, j + 1), node(i, j + 1)]
                if j + 1 == jm:                                            # the row just below the slit
                    q = [q[0], q[1], lower.get(i + 1, q[2]), lower.get(i, q[3])]
                cells += [[q[0], q[1], q[2]], [q[0], q[2], q[3]]]
        cells = np.array(cells, dtype=np.int64)
    else:
        x, cells = np.zeros((0, 2)), np.zeros((0, 3), dtype=np.int64)
    return mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 1, shape=(2,))), x)

class PhaseField:
    """plane strain; split = "amor" (psi+ = K/2 <tr eps>+^2 + mu dev eps : dev eps) or "none" (psi+ = psi).
    Bottom: u_y = 0 (and u_x = 0 at the corner (0, 0)); top: u_y = U (u_x free)."""
    def __init__(self, msh, E, nu, Gc, ell, split="amor", k=1e-6, d_one=None):
        """d_one: optional marker x -> bool of nodes held at d = 1 (a notch drawn into the damage field instead of the mesh)"""
        self.msh, self.Gc, self.ell = msh, Gc, ell
        mu, lm = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu)); K = lm + 2 * mu / 3
        V = self.V = fem.functionspace(msh, ("Lagrange", 1, (2,))); W = self.W = fem.functionspace(msh, ("Lagrange", 1))
        Q = fem.functionspace(msh, basix.ufl.quadrature_element(msh.basix_cell(), value_shape=(), degree=2))
        self.u, self.d, self.H, self.psi = fem.Function(V), fem.Function(W), fem.Function(Q), fem.Function(Q)
        self.H_n = np.zeros_like(self.H.x.array)
        dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 2})
        eps = lambda w: ufl.sym(ufl.grad(w))
        sig = lambda w: lm * ufl.tr(eps(w)) * ufl.Identity(2) + 2 * mu * eps(w)
        g = (1 - self.d)**2 + k
        du, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        dd, q = ufl.TrialFunction(W), ufl.TestFunction(W)
        self.au = fem.form(g * ufl.inner(sig(du), eps(v)) * dx)
        self.Lu = fem.form(ufl.dot(fem.Constant(msh, np.zeros(2)), v) * dx)
        self.ad = fem.form((Gc / ell + 2 * self.H) * dd * q * dx + Gc * ell * ufl.inner(ufl.grad(dd), ufl.grad(q)) * dx)
        self.Ld = fem.form(2 * self.H * q * dx)
        e = eps(self.u); tr = ufl.tr(e)
        e3 = ufl.as_matrix([[e[0, 0], e[0, 1], 0], [e[1, 0], e[1, 1], 0], [0, 0, 0]])     # plane strain: eps_zz = 0
        dev3 = e3 - tr / 3 * ufl.Identity(3)
        psi_full = lm / 2 * tr**2 + mu * ufl.inner(e, e)
        psi_plus = K / 2 * ((tr + abs(tr)) / 2)**2 + mu * ufl.inner(dev3, dev3) if split == "amor" else psi_full
        self.psi_expr = fem.Expression(psi_plus, Q.element.interpolation_points)
        fd = lambda f: mesh.locate_entities_boundary(msh, 1, f)
        corner = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0))
        self.U = fem.Constant(msh, 0.0)
        self.bcu = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(1), 1, fd(lambda x: np.isclose(x[1], 0))), V.sub(1)),
                    fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(0), 0, corner), V.sub(0)),
                    fem.dirichletbc(self.U, fem.locate_dofs_topological(V.sub(1), 1, fd(lambda x: np.isclose(x[1], 1))), V.sub(1))]
        self.F_form = fem.form(g * sig(self.u)[1, 1] * dx)                 # consistent reaction: virtual work with w = (0, y)
        self.G_form = fem.form(Gc / 2 * (self.d**2 / ell + ell * ufl.inner(ufl.grad(self.d), ufl.grad(self.d))) * dx)
        self.Eel_form = fem.form(g * psi_full * dx)
        self.band = fem.Constant(msh, 1e9)
        X = ufl.SpatialCoordinate(msh)
        inside = ufl.conditional(ufl.lt(abs(X[1] - 0.5), self.band), 1.0, 0.0)
        self.Gb_form = fem.form(inside * Gc / 2 * (self.d**2 / ell + ell * ufl.inner(ufl.grad(self.d), ufl.grad(self.d))) * dx)
        self.bcd = [] if d_one is None else [fem.dirichletbc(1.0, fem.locate_dofs_geometrical(W, d_one), W)]
        def mk(a, Vs):
            A = create_matrix(a); kk = PETSc.KSP().create(comm); kk.setOperators(A); kk.setType("preonly")
            kk.getPC().setType("lu"); kk.getPC().setFactorSolverType("mumps"); return A, kk, create_vector(Vs)
        self.Au, self.ku, self.bu = mk(self.au, V)
        self.Ad, self.kd, self.bd = mk(self.ad, W)

    def _solve(self, A, k, b, a, L, bcs, f):
        A.zeroEntries(); assemble_matrix(A, a, bcs=bcs); A.assemble()
        with b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(b, L); apply_lifting(b, [a], [bcs])
        b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
        k.solve(b, f.x.petsc_vec); f.x.scatter_forward()

    def step(self, U, tol=1e-4, maxit=300):
        """one load step: alternate u-solve, H = max(H_n, psi+), d-solve until max|d - d_previous| < tol.
        Returns (converged, iterations). The history is committed in any case (the caller must check the flag)."""
        self.U.value = U
        for it in range(1, maxit + 1):
            d_old = self.d.x.array.copy()
            self._solve(self.Au, self.ku, self.bu, self.au, self.Lu, self.bcu, self.u)
            self.psi.interpolate(self.psi_expr)
            self.H.x.array[:] = np.maximum(self.H_n, self.psi.x.array)
            self._solve(self.Ad, self.kd, self.bd, self.ad, self.Ld, self.bcd, self.d)
            err = comm.allreduce(float(np.abs(self.d.x.array - d_old).max()), op=MPI.MAX)
            if err < tol:
                break
        self.H_n = self.H.x.array.copy()
        return err < tol, it

    def force(self):
        return comm.allreduce(fem.assemble_scalar(self.F_form), op=MPI.SUM)

    def crack_energy(self, band=None):
        """regularised crack energy; band = w: only the strip |y - 0.5| < w (the crack band, without the diffuse damage)"""
        if band is None:
            return comm.allreduce(fem.assemble_scalar(self.G_form), op=MPI.SUM)
        self.band.value = band
        return comm.allreduce(fem.assemble_scalar(self.Gb_form), op=MPI.SUM)

    def elastic_energy(self):
        return comm.allreduce(fem.assemble_scalar(self.Eel_form), op=MPI.SUM)

    def d_max(self):
        return comm.allreduce(float(self.d.x.array.max()), op=MPI.MAX)

## Q1 — Theory (20 pts)

**(a) (7)** For one branch $\dot\varepsilon^v=(\varepsilon-\varepsilon^v)/\tau$, derive the backward-Euler step and the exponential step for a strain that is linear in time over
$[t_n,t_{n+1}]$, both in the form $\varepsilon^v_{n+1}=a\varepsilon^v_n+(1-b)\varepsilon_{n+1}+(b-a)\varepsilon_n$. Show that both tend to $a=b=1$ as $\Delta t\to0$ and explain why that is the
correct instantaneous (elastic) step.

**(b) (7)** Standard linear solid: $E(t)=E_\infty+E_1e^{-t/\tau}$. Derive the creep compliance $J(t)=\frac1{E_\infty}-\Big(\frac1{E_\infty}-\frac1{E_0}\Big)e^{-t/\lambda}$ and show that the
retardation time is $\lambda=\tau E_0/E_\infty$. Is creep slower or faster than relaxation? Check your formula against `Prony.creep` for $E_\infty=10$, $E_1=5$, $\tau=1$.

**(c) (6)** For AT2 in a homogeneous bar derive $\sigma_{\max}=\frac{3\sqrt3}{16}\sqrt{EG_c/\ell}$ and the damage at the peak. A structure made of this material has a crack
much longer than $\ell$. Does its failure load depend on $\ell$? And a structure with no crack and no notch?

> *your answer*

In [ ]:
# TODO: Q1(b) — your closed-form J(t) of the standard linear solid against Prony.creep

## Q2 — Viscoelasticity in 2D (25 pts)

The lab's tube ($r_i=1$, $r_o=2$, $8\times16$ curved cells) and Prony material.

**(a) (8)** Creep under $p=1$ to $t=5$ with EXP and BE for $\Delta t$ = 0.2, 0.1, 0.05, 0.025. Print the maximum relative error against $pJ(t)\tilde u$ and the observed order
$\log_2(e_{\Delta t}/e_{\Delta t/2})$. Why is it pointless to refine the mesh for this study?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) — time-step study, both schemes

> *your table and answer (Q2a)*

**(b) (8)** Relaxation: hold $u_r(r_i)=u_0=0.01$ from $t=0$ (`control="displacement"`) and read the pressure from the consistent reaction. Compare with the exact
$p(t)=u_0E(t)/\tilde u$ for EXP and BE ($\Delta t=0.2$), with and without the elastic step at $0^+$. Which combination is exact, and why?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) — relaxation under displacement control

> *your answer (Q2b)*

**(c) (9)** Give the branches different Poisson ratios: $\nu_\infty=0.3$, $\nu_1=\nu_2=0.45$ (`branch_nu=[0.3, 0.45, 0.45]`) — the viscous branches are nearly incompressible.
Run the creep test (EXP, $\Delta t=0.05$). (i) Check $u_r(0^+)$ and $u_r(t\to\infty)$ against elastic solutions — with which moduli? (ii) How far is $u_r(t)$ from the
"correspondence" guess $pJ(t)\tilde u(\nu=0.3)$ at $t=0.5$ and $t=2$? (iii) Why can no single function of time describe this tube?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) — branches with different Poisson ratios

> *your answer (Q2c)*

## Q3 — Phase field (35 pts)

The lab's slit specimen (mode I, $E=210$ GPa, $\nu=0.3$, $G_c=2.7\times10^{-3}$ kN/mm, 80 steps to $U=8\,\mu$m). The runs take a few minutes on Colab.

**(a) (12)** Mesh study at $\ell=0.04$: $h=0.02$ and $h=0.01$. **(b) (10)** Length-scale study at $h=\ell/2$: $\ell=0.04$ and $\ell=0.02$. For each run print the peak force,
the crack energy (total and in the band $|y-\tfrac12|<5\ell$) over $G_ca$, whether every step converged, and the largest number of staggered iterations.
Explain: what changes $F_{\max}$ in (a), what changes it in (b)? Which of the two energies would you compare with $G_c$, and what remains?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a)(b) — three runs of the slit specimen

> *your table and explanation (Q3ab)*

**(c) (13)** The split. A homogeneous material point in **uniaxial strain** (plane strain, $\varepsilon_{yy}=\varepsilon$, all other strains 0), AT2 with $\ell=0.02$: compute the damage and
the stress $\sigma_{yy}$ for $-0.06\le\varepsilon\le0.06$ with $\psi^+=\psi$ (no split) and with the Amor split ($\psi^+=\tfrac K2\langle\operatorname{tr}\boldsymbol\varepsilon\rangle_+^2+\mu\,\mathbf e:\mathbf e$, plane strain). Use
the hybrid rule of the lecture: $d$ from $\psi^+$, $\sigma=g(d)\,\boldsymbol\sigma_0(\boldsymbol\varepsilon)$. (i) Plot $\sigma_{yy}(\varepsilon)$ for both. (ii) What happens in compression without the split, and
what with Amor? (iii) Where does the hybrid rule become inconsistent, and which formulation would fix it?

In [ ]:
# TODO: Q3(c) — the split at a material point (NumPy)
import numpy as np, matplotlib.pyplot as plt

> *your answer (Q3c)*

## Q4 — Debugging an LLM-generated phase-field script (20 pts)

A student asked an LLM for "a short DOLFINx 0.11 script for a 1D AT2 bar, $x\in[0,1]$, fixed at $x=0$, pulled at $x=1$, with a slightly weaker zone in the middle so
that the crack forms there; load until it breaks ($U=4$), unload to zero and reload to $U=4$, and plot the force". The script below runs and prints a force for every step. It has
**four** defects.

**(a) (4)** Run it as it is and record the output.

**(b) (8)** List the four defects. For each, name the test that exposes it (the homogeneous strength, the 1D profile energy, the force after complete fracture,
unloading and reloading, …).

**(c) (8)** Correct the script in the second cell and verify it: the peak force against $\sigma_{\max}$ of the weak zone, the force after fracture, the reloading stiffness, and
the crack energy against $G_c$ of the weak zone.

In [ ]:
%%fenicsx -np 1
# LLM-generated (as received)
import numpy as np, ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
E, Gc, ell, k = 1.0, 1.0, 0.05, 1e-6
msh = mesh.create_interval(comm, 400, [0.0, 1.0])
x = ufl.SpatialCoordinate(msh)[0]
Gcx = Gc * (1 - 0.1 * ufl.conditional(ufl.lt(abs(x - 0.5), 0.05), 1.0, 0.0))     # weak zone in the middle
V = fem.functionspace(msh, ("Lagrange", 1)); W = fem.functionspace(msh, ("Lagrange", 1))
Q = fem.functionspace(msh, basix.ufl.quadrature_element(msh.basix_cell(), value_shape=(), degree=2))
u, d, H, psi = fem.Function(V), fem.Function(W), fem.Function(Q), fem.Function(Q)
g = (1 - d)**2 + k
du, v = ufl.TrialFunction(V), ufl.TestFunction(V)
dd, q = ufl.TrialFunction(W), ufl.TestFunction(W)
Uc = fem.Constant(msh, 0.0)
left = fem.locate_dofs_geometrical(V, lambda X: np.isclose(X[0], 0)); right = fem.locate_dofs_geometrical(V, lambda X: np.isclose(X[0], 1))
bcs = [fem.dirichletbc(0.0, left, V), fem.dirichletbc(Uc, right, V)]
opts = {"ksp_type": "preonly", "pc_type": "lu"}
pu = LinearProblem(g * E * du.dx(0) * v.dx(0) * ufl.dx, fem.Constant(msh, 0.0) * v * ufl.dx, bcs=bcs, u=u, petsc_options_prefix="u_", petsc_options=opts)
pd = LinearProblem((Gcx * ell + 2 * H) * dd * q * ufl.dx + Gcx / ell * dd.dx(0) * q.dx(0) * ufl.dx, H * q * ufl.dx, u=d, petsc_options_prefix="d_", petsc_options=opts)
psi_expr = fem.Expression(E / 2 * u.dx(0)**2, Q.element.interpolation_points)
F_form = fem.form(E * u.dx(0) * ufl.dx)                                          # bar of length 1: the mean stress = the force

loads = np.r_[np.linspace(0, 4.0, 81)[1:], np.linspace(4.0, 0.0, 9)[1:], np.linspace(0.0, 4.0, 9)[1:]]
for n, U in enumerate(loads):
    Uc.value = U
    for it in range(200):
        d_old = d.x.array.copy()
        pu.solve(); psi.interpolate(psi_expr); H.x.array[:] = psi.x.array; pd.solve()
        if np.abs(d.x.array - d_old).max() < 1e-6:
            break
    if n % 8 == 7:
        print(f"step {n + 1:3d}: U = {U:.3f}  F = {fem.assemble_scalar(F_form):.4f}  d_max = {d.x.array.max():.3f}  iterations {it + 1}")

In [ ]:
%%fenicsx -np 1
# TODO: Q4(c) — the corrected script and its verification

> *your list of defects and tests (Q4b), and your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …